<div style="text-align: center"><img src="https://telematica.usm.cl/wp-content/themes/telematica-web/assets/img/header-usm.png" height="75"></div>

# Tarea 4: Mensajería CoAP y MQTT
### Asignatura: Redes de Sensores - TEL329 - 2026
#### Docente y autor: Berioska Contreras Vargas - berioska.contreras@usm.cl

###### Este contenido es parte de la asignatura TEL329 de UTFSM. Las tareas son un complemento a las lecciones teóricas. El objetivo es simular y emular redes de sensores constreñidos.

## Protocolo CoAP

El protocolo CoAP (Constrained Application Protocol) es un protocolo de la capa de aplicación diseñado para ser ligero y propuesto por la Fuerza de Trabajo de Ingeniería de Internet (IETF) el año 2014. Utiliza UDP como protocolo de transporte subyacente, lo que reduce la sobrecarga en comparación con TCP. CoAP consiste de un modelo de comunicación cliente-servidor, semejante a HTTP, y admite métodos como GET, POST, PUT y DELETE para interactuar con recursos específicos. CoAP se integra con direccionamiento IPv6 y con el mecanismo de seguridad DTLS (Datagram Transport Layer Security) para proporcionar confidencialidad en las comunicaciones. 

La mayor parte del código que usaremos se encuentra en $(CONTIKI)/examples/coap/.

| Fuente | Referencia | 
| - | - |
| [ContikiNG Docs](https://docs.contiki-ng.org/en/develop/doc/programming/CoAP.html) | Contiki Next Generation Documentation - CoAP |
| [ContikiNG Repo](https://github.com/contiki-ng/contiki-ng) | Contiki-NG: The OS for Next Generation IoT Devices |
| [RFC 7252: 2014](https://www.rfc-editor.org/rfc/rfc7252) | The Constrained Application Protocol (CoAP) |

### Servidor CoAP de ContikiNG

El código $coap-example-server.c$ implementa un simple servidor CoAP con varios recursos. Declara recursos CoAP externos p.e. res_hello, res_mirror, etc. que están definidos en archivos separados (examples/coap/coap-example-server/resources). Estos recursos representan distintas funcionalidades o datos expuestos por el servidor.

La llamada de recursos es posible a través de la **activación de cada recurso** lo que permite resolver o mapear cada recurso a una ruta $URI$ específica p.e. test/hello. El código $coap-example-server.c$ está a la espera eventos p.e. presionar el botón, y así reaccionar a ellos con acciones como ejecutar una notificación o actualización de variable. Inicie el servidor CoaP/Erbium de ContikiNG según los Pasos 1 y 2. Verifique que el proceso tiene asociada una dirección IPv6 local y global disponible a través de un túnel serial p.e. tunslip.

In [ ]:
# Paso 1 - Servidor CoAP ContikiNG
$ cd contiki-ng/examples/coap/coap-example-server/coap-example-server.c
$ sudo make distclean
$ sudo make target=native
$ sudo /build/native/coap-examples-server.native

# Paso 2 - Resultado esperado
...
[INFO: Main      ] Tentative link-local IPv6 address: fe80::302:304:506:708
[INFO: Native    ] Added global IPv6 address fd00::302:304:506:708
[INFO: App       ] Starting Erbium Example Server
...

### Cliente CoAP de ContikiNG

El código $coap-example-client.c$ implementa el cliente CoAP que periódicamente envía peticiones al servidor Coap y que envía notificaciones cada vez que se presione un botón del dispositivo. Además, define un servidor Erbium específico p.e. SERVER_EP "coap://[fe80::302:304:506:708]". Utiliza un arreglo de rutas de servicio que contienen URIs de recursos del servidor. El intervalo de envío de notificaciones es fijo p.e. Toggle_Interval. Cuando el temporizador expira, envía un petición POST con el contenido $Toggle!$


En general, el cliente permite que un usuario envíe distintas peticiones de recuros hacia el servidor a través de un evento específico p.e. presionar un botón, para luego desplegar la respuesta recibida. Inicie el cliente CoaP de ContikiNG según los Pasos 3 a 6. Verifique que el código respuesta sea consistente con el método y recurso consultado.

| Fuente | Referencia | 
| - | - |
| [CoAP-CLI](https://www.npmjs.com/package/coap-cli) | CoAP-CLI built on node.js |

In [ ]:
# Paso 3 - Cliente Coap ContikiNG
$ cd contiki-ng/examples/coap/coap-example-client/coap-example-client.c
$ sudo make distclean
$ sudo make target=native
$ sudo /build/native/coap-examples-client.native

# Paso 4 - Resultado esperado
...
--Toggle timer--
coap://[fe80::302:304:506:708]:5683
...

# Paso 5 - Instalar utilitarios de CoAP
$ sudo apt-get install -y npm
$ sudo npm install coap-cli -g

# Paso 6 - Consultar recursos
$ coap get coap://[fe80::302:304:506:708]:5683/.well-known/core
(2.05)
$ coap get coap://[fe80::302:304:506:708]:5683/led
(4.04)
$ coap get coap://[fe80::302:304:506:708]:5683/led?color=green mode=off
(4.04)
...

## Protocolo MQTT

El protocolo MQTT (Message Queue Telemetry Transport) es un protocolo diseñado sobre la capa de transporte TCP/IP, que le permite una integración directa con el protocolo TLS (Transport Layer Security) para una comunicación cifrada entre dispositivos. MQTT fue desarrollado originalmente por IBM a finales de 1990, y en 2014 se transformó en estándard abierto OASIS. El protocolo MQTT es asincrónico y separa la mensajería de envío y recepción en tiempo y espacio, siendo escalable para entornos de red inestables.

La mayor parte del código que usaremos se encuentra en $(CONTIKI)/examples/mqtt-client

| Fuente | Referencia | 
| - | - |
| [MQTT IBM](https://developer.ibm.com/articles/iot-mqtt-why-good-for-iot/) | Michael Yuan - Getting to Know MQTT |
| [MQTT OASIS v3,1](https://docs.oasis-open.org/mqtt/mqtt/v3.1.1/mqtt-v3.1.1.html) | OASIS Standard MQTT v3.1 |
| [MQTT ContikiNG](https://docs.contiki-ng.org/en/develop/doc/tutorials/MQTT.html) | ContikiNG MQTT Documentation |
| [MQTT Mosquitto](https://mosquitto.org/) | An OpenSource MQTT Broker |

### Broker MQTT de ContikiNG

El componente central de MQTT es el Broker, este actúa como un intermediario entre los clientes que publican mensajes (publishers) y los clientes que se suscriben a ellos (subscribers). Recibe los mensajes publicados por los clientes, incluyendo el tema al que pertenecen y la "calidad de servicio" (QoS) deseada. Filtra los mensajes en función de las suscripciones de los clientes. Solo entrega los mensajes a los clientes que se han suscrito a los temas correspondientes. Mantiene una lista de suscripciones de los clientes, incluyendo los temas a los que se han suscrito. Además, puede implementar mecanismos de seguridad para autenticar a los clientes y autorizar el acceso a los temas.

Utilizaremos el servicio Broker del proyecto abierto Mosquitto según los Pasos 7 y 8. Verifique que el servicio está escuchando en el 

| Fuente | Referencia | 
| - | - |
| [Mosquitto](https://mosquitto.org/) | An OpenSource MQTT Broker |
| [ActiveMQ](https://activemq.apache.org/) | An OpenSource MQTT Broker by Apache |

In [ ]:
# Paso 7 - Instalar Mosquitto broker & clients
$ sudo apt-get install -y mosquitto mosquitto-clients
$ sudo vi /etc/mosquitto/mosquitto.conf
...
listener 1883
allow_anonymous true
...
$ sudo systemctl status|stop|start mosquitto

#Paso 8 - Iniciar broker y resultado esperado
$ mosquitto -v
...
1728432200: Opening ipv4 listen socket on port 1883.
1728432200: Opening ipv6 listen socket on port 1883.
1728432200: mosquitto version 2.0.11 running
...

### Suscripción/Publicación MQTT

El código $mqtt-client.c$  implementa un cliente MQTT para ContikiNG que se conecta a un broker MQTT (como Mosquitto o IBM Watson IoT Platform) y publica datos de sensores periódicamente. Además, define un servidor Broker específico p.e. SERVER_EP $MQTT\_CLIENT\_BROKER\_IP\_ADDR "fd00::1"$. Maneja eventos MQTT como conexión, desconexión, publicación y suscripción.  Publica datos del dispositivo (como la plataforma, la placa, el número de secuencia y el tiempo de actividad) en un tema MQTT. Se suscribe a un tema para recibir comandos y controlar los LEDs del dispositivo.

En general, el código proporciona un cliente MQTT básico para ContikiNG, además el código es modular y extensible, lo que permite adaptarlo a diferentes necesidades y plataformas. Inicie el suscriptor/publicador MQTT según los Pasos 9 a 11. Verifique que el proceso tiene asociada una dirección IPv6 local y global disponible a través de un túnel serial p.e. tunslip.


In [ ]:
# Paso 9 - Suscripción de tema general desde terminales
$ mosquitto_sub -h localhost -t "#"
...
Telematica
$ mosquitto_pub -h localhost -t "usm/cssj/delo/ictel/json" -m "Telematica"
...
1728432939: Received PUBLISH from auto-E17C575F-1512-7E3D-F0F8-E67834553271 (d0, q0, r0, m0, 'usm/cssj/delo/ictel/json', ... (10 bytes))
1728432939: Sending PUBLISH to auto-2A2AC4C8-1112-55E2-6ED9-A75678A4F29C (d0, q0, r0, m0, 'usm/cssj/delo/ictel/json', ... (10 bytes))

#Paso 10 - Cliente MQTT ContikiNG y resultado esperado
$ cd contiki-ng/examples/mqtt-client/mqtt-client.c
$ sudo make distclean
$ sudo make target=native
$ sudo ./mqtt-client.native
...
opened tun device ``/dev/tun0''
tun0: flags=4305<UP,POINTOPOINT,RUNNING,NOARP,MULTICAST>  mtu 1500
        inet 127.0.1.1  netmask 255.255.255.255  destination 127.0.1.1
        inet6 fd00::1  prefixlen 64  scopeid 0x0<global>
        inet6 fe80::293b:7f98:adc5:19c0  prefixlen 64  scopeid 0x20<link>
        unspec 00-00-00-00-00-00-00-00-00-00-00-00-00-00-00-00  txqueuelen 500  (UNSPEC)
        RX packets 0  bytes 0 (0.0 B)
        RX errors 0  dropped 0  overruns 0  frame 0
        TX packets 0  bytes 0 (0.0 B)
        TX errors 0  dropped 0 overruns 0  carrier 0  collisions 0

[INFO: Main      ] Starting Contiki-NG-develop/v4.9-70-g3695d415e-dirty
[INFO: Main      ] - Routing: RPL Lite
[INFO: Main      ] - Net: tun6
[INFO: Main      ] - MAC: nullmac
[INFO: Main      ] - 802.15.4 PANID: 0xabcd
[INFO: Main      ] - 802.15.4 Default channel: 26
[INFO: Main      ] Node ID: 1800
[INFO: Main      ] Link-layer address: 0102.0304.0506.0708
[INFO: Main      ] Tentative link-local IPv6 address: fe80::302:304:506:708
[INFO: Native    ] Added global IPv6 address fd00::302:304:506:708
MQTT Client Process

## Actividades 1, 2 y 3:

| N° | Actividad | 
| - | - |
| 1 | Implemente una pequeña red WSN simulada que integre los protocolos RPL Lite y Coap, considerando un nodo raíz y 5 nodos clientes que notifiquen sólo un recurso p.e. temperatura generada de manera aleatoria. Hint random_rand(). Presente su código adaptado y visualice sus resultados en el simulador Cooja o Terminal.  |
| 2 | Implemente una pequeña red WSN simulada que integre los protocolos RPL Lite y MQTT, considerando un nodo raíz y 5 nodos clientes que notifiquen sólo un tema p.e. temperatura generada de manera aleatoria. Hint random_rand(). Presente su código adaptado y visualice sus resultados en el simulador Cooja o Terminal.  |
| 3 | Realice una captura de tráfico de la mensajería generada por el protocolo CoaP y el protocolo MQTT de los puntos 1 y 2. Luego, extraiga los tiempos de respuesta de una muestra representativa de mensajes de cada uno. Presente los datos en una tabla que resuma el tiempo de respuesta de consulta de cada protocolo, analice sus resultados y explíquelos de manera sucinta. |

## Sensores en la Nube

La conexión de los sensores WSN a la nube computacional ofrece una serie de beneficios que van desde el almacenamiento y procesamiento de datos hasta la gestión remota y el análisis avanzado. Esto permite desarrollar aplicaciones más sofisticadas, eficientes y escalables en una variedad de campos. Por ejemplo: Agricultura de precisión, Monitorización ambiental, Ciudades inteligentes, entre otros.

Los servicios en la nube se pueden clasificar en tres grandes categorías:  Infraestructura como servicio (IaaS), Plataforma como servicio (PaaS) y  Software como servicio (SaaS). 

IaaS proporciona acceso a recursos informáticos fundamentales como servidores, almacenamiento, redes y sistemas operativos a través de Internet. Ejemplos: Amazon Web Services (AWS) EC2, Microsoft Azure Virtual Machines, Google Compute Engine.

PaaS proporciona un entorno de desarrollo e implementación de aplicaciones en la nube. Ejemplos: AWS Elastic Beanstalk, Google App Engine, Microsoft Azure App Service.

SaaS proporciona acceso a aplicaciones de software a través de Internet. Ejemplos: Gmail, Salesforce, Dropbox, Google Workspace.

| Fuente | Referencia | 
| - | - |
| [NIST-SP-800-145](https://www.nist.gov/publications/nist-definition-cloud-computing) | Peter M. Mell and Timothy Grance. (2011). The NIST definition of cloud computing. National Institute of Standards and Technology. |
| [NIST-IR-8200](https://nvlpubs.nist.gov/nistpubs/ir/2018/NIST.IR.8200.pdf) | Editors Mike Hogan and Ben Piccarreta. (2018). Interagency Report on the Status of International Cybersecurity Standardization for the Internet of Things (IoT).|

## Microsoft IoT Hub

Microsoft IoT Hub, también conocido como Azure IoT Hub, es un servicio administrado en la nube que actúa como un centro de mensajes para la comunicación entre una aplicación de Internet de las Cosas (IoT) y los dispositivos conectados. IoT Hub permite conectar millones de dispositivos de manera segura y confiable, facilitancdo la comunicación tanto desde los dispositivos hacia la nube (telemetría) como desde la nube hacia los dispositivos (comandos y notificaciones). Puede manejar millones de dispositivos y eventos por segundo, lo que lo hace adecuado para grandes implementaciones de IoT.

Nuestro objetivo es conectar sensores de Contiki-NG en la plataforma Azure IoT Hub. 

| Fuente | Referencia | 
| - | - |
| [Azure IoT Hub](https://learn.microsoft.com/es-es/azure/iot-hub/iot-concepts-and-iot-hub) | ¿Qué es Microsoft IoT Hub? |

## Actividad 4

El objetivo es desarrollar o implementar las siguientes aplicaciones:

| Aplicación | Descripción | 
| - | - |
| Sensor | Mote de Contiki-NG que envía información sensada |
| Enrutador | Router 6LowPAN en Contiki-NG |
| Interfaz | Middleware que recibe datos sensados y los reeenvía a la nube |
| Nube | Azure IoT Hub notifica datos a los dispositivos suscritos |

La implementación consiste de los siguientes pasos: preparar la configuración de Azure IoT Hub utilizando llaves de acceso simétrico, desarrollar las aplicaciones para el enrutador y sensores Contiki-NG y middleware (p.e. websense). Finalmente, se comprobarán todos los programas.

El sensor ejecutará un servidor web que estará a la espera de peticiones entrantes al sensor (Ver Paso 1 y Paso 2). Mientras, la aplicación del enrutador implementará el programa $rpl-border-router" ya estudiada.

La interfaz $middleware$ solicitará datos a los sensores desde Contiki-NG y luego, los enviará al IoT Hub de Azure (Ver Pasos 3, 4 y 5).

La última aplicación se suscribirá al Azure IoT Hub para obtener los datos enviados por los sensores de Contiki-NG. Así, el programa consumirá los datos sensados que envió la interfaz $middleware$ (Ver Pasos 6, 7 y 8).

| Fuente | Referencia | 
| - | - |
| [Practical Contiki-NG](https://link.springer.com/book/10.1007/978-1-4842-3408-2) | Kurniawan, Agus. (2018). Practical Contiki-NG: Programming for Wireless Sensor Networks (p. 284). Apress. |

In [ ]:
# Paso 1 - Makefile del Sensor
CONTIKI_PROJECT = websense-Cloud
all: $(CONTIKI_PROJECT)
CONTIKI = ../.. 
PROJECT_SOURCEFILES += http-simple.c 
include $(CONTIKI)/Makefile.include

In [ ]:
# Paso 2 Adaptación Websense del Sensor
static
PT_THREAD(generate_routes(struct httpd_state *s)){
    char buff[35];
    PSOCK_BEGIN(&s->sout);
    //SEND_STRING(&s->sout, TOP);
    
    int temperature = 15 + rand() % 25;
    int humidity = 80 + rand() % 10:

    sprintf(buff,"{\"temp\":%u,\"hum\":%u}", temperature, humidity);
    printf("send json to requester\n");
    SEND_STRING(&s->sout,buff);
    //SEND_STRING(&s->sout,BOTTOM);
    PSOCK_END(&s->sout);
}

In [ ]:
# Paso 3 - Preparar Middleware con Node.js
$ npm init 
$ npm install azure-iot-device azure-iot-device-mqtt express --save
$ vi middleware-azure.js

# Paso 4 - Contenido de middleware-azure.js
'use strict';
var express = require('express');
var request = require('request');
var app = express();
var http = require('http').Server(app);
var clientFromConnectionString=require('azure-iot-device-mqtt').clientFrom-ConnectionString;
var Message = require('azure-iot-device').Message;
var connectionString = 'HostName={youriothostname};DeviceID={youriotid};SharedAccessKey={yourdevicelkey}';
var mydevideId = 'contiki-ng-01'
var client = clientFromConnection-String(connectionString);

app.get('/', function(req,res){
    res.send('Wensense Azure Cloud');
});

http.listen(3000, function(){
    console.log('listening on *:3000');
    console.log('websense azure Cloud was started');
});

function printResultFor(op){
    return function printResult(err,res){
        if(err) 
            console.log(op + 'error' + err.toString());
        if(res) 
            console.log(op + 'status' + res.constructor.name);
    };
};

var connectCallback = function(err){
     if(err) {
         console.log('Could not connect:' + err);
     } else {+
        console.log('Clinet connected');     
        setInterval(function(){
            request.get('http://[IPv6-mote-websense]/',function(err,res,body){  
                if(err){
                    console.log(err);
                    return;
                }
                var obj = JSON.parse(body);
                console.log(obj);
                var temperature = obj.temp;
                var humidity = obj.hum;
                var data = JSON.stringi-fy({devideId:mydecideId,temperature:temperature,humidity:humidity});
                var message = new Message(data);
                message.proper-ties.add('temperatureAlert',(temperature > 30)?'true':'false');
                console.log("Sending message:" + message.getData());
                client.sendEvent(message,printResultFor('send'));
            });
        },3000);
        }
};

client.open(connectCallback);
console.log('Contiki-NG Azure Middleware started.');

# Paso 5 - Comprobación middleware-azure.js
$ node middleware-azure.js

In [ ]:
# Paso 6 - Preparar Middleware con Node.js
$ npm init 
$ npm install azure-event-hubs --save
$ vi azure-sensor-subscriber.js

# Paso 7 - Contenido de azure-sensor-subscriber.js
'use strict';
var EventHubClient = require('azure-event-hubs').Client;
var connectionString = 'HostName={youriothostname},DeviceId=myiotId;SharedAccessKey={yourdevicekey}}';

var printError = function(err){
    console.log(err.message);
};

var printMessage = function(message){
    console.log('Message received:');
    console.log(JSON.stringify(message.body));
    console.log('');
};

var client = EventHubClient.fromConnectionString(connectionString);
    client.open()
        .then(client.getPartitionIds.bind(client))
        .then(function (partitionIds){
            return partitionIds.map(function(partitionId){
                return client.createReceiver('$Default',partitionId,
                                             {'startAfter-Time':Date.now()}).then(function(receiver){
                                                console.log('Created partition receiver:'+ partitionId)
                                                receiver.on('errorReceived',printError);
                                                receiver.on('message',printMessage);
                                             });                                   
            });
        })
    .catch(printError);

# Paso 8 - Comprobación azure-sensor-subscriber.js
$ node azure-sensor-subscriber.js

## Actividad 4 - Preguntas:
 
La actividad 4 implica la ejecución de Contiki-NG ya sea en un host local o en la nube de Microsoft Azure. 

| N° | Pregunta | 
| - | - |
| 4.a | ¿Cuál es el resultado presentado por la interfaz middleware? Proporcione evidencias de su implementación y describa su planteamiento. |
| 4.b | ¿Cuál es el resultado presentado por el sensor o Mote? Proporcione evidencias de su implementación y describa su planteamiento. |
| 4.c | ¿Cuál es el resultado presentado por el suscriptor de Azure Iot Hub? Proporcione evidencias de su implementación y describa su planteamiento. |

Notar: El código presentado está basado en el trabajo de Kurniawan (2018) y puede ser adaptado de acuerdo al escenario o caso de uso que determine, siendo necesario publicar dicha adaptación en la plataforma GitHub, o equivalente, para la evaluación.

## Fin de la Tarea!